# Week 3: Rapid KL GTFS Data Pipeline

This notebook implements the selected structured/tabular pathway (B) for the Rapid Bus KL static GTFS feed. It downloads the public ZIP, inspects selected tables, creates a joined route-stop lookup, validates its keys and row counts, writes lineage manifests, and reruns the pipeline to compare fingerprints.

The feed is static and the provider reports about 2% of Rapid Bus KL trips missing from `stop_times.txt`. Do not use this notebook to report live arrivals or infer service absence from missing records.

**Run order:** Cells 1–3, then Part B preparation and validation, then Part C pipeline and repeatability cells. All code runs in Google Colab.


## Cell 1 — Initialize the Colab workspace

In [ ]:
from pathlib import Path
import hashlib, json, shutil, time, zipfile, gzip
from datetime import datetime, timezone
from urllib.request import urlopen
from io import BytesIO
import pandas as pd

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_URL = "https://api.data.gov.my/gtfs-static/prasarana?category=rapid-bus-kl"
API_DOC = "https://developer.data.gov.my/realtime-api/gtfs-static"
LICENSE = "CC BY 4.0"
print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)


## Cell 2 — Ingest and inspect the source

In [ ]:
response = urlopen(SOURCE_URL, timeout=60)
source_bytes = response.read()
source_path = RAW_DIR / "rapid-bus-kl-gtfs.zip"
source_path.write_bytes(source_bytes)
source_sha256 = hashlib.sha256(source_bytes).hexdigest()

with zipfile.ZipFile(BytesIO(source_bytes)) as zf:
    bad_member = zf.testzip()
    if bad_member is not None:
        raise ValueError(f"ZIP CRC check failed: {bad_member}")
    available = set(zf.namelist())
    required = ["routes.txt", "trips.txt", "stops.txt", "stop_times.txt", "calendar.txt"]
    missing = sorted(set(required) - available)
    if missing:
        raise ValueError(f"Required GTFS tables missing: {missing}")
    tables = {name: pd.read_csv(zf.open(name), dtype=str, keep_default_na=False) for name in required}
    agency = pd.read_csv(zf.open("agency.txt"), dtype=str, keep_default_na=False)

print("HTTP status:", response.status)
print("ZIP bytes:", len(source_bytes), "SHA-256:", source_sha256)
print("Files:", sorted(available))
for name, frame in tables.items():
    print(f"{name}: {len(frame):,} rows; columns={frame.columns.tolist()}")
    display(frame.head(2))
print("Agency sample:")
display(agency.head(2))


## Cell 3 — Prepare the Part B lookup and validate it

In [ ]:
routes = tables["routes.txt"]
trips = tables["trips.txt"]
stops = tables["stops.txt"]
stop_times = tables["stop_times.txt"]
calendar = tables["calendar.txt"]

# Structural and referential checks before joining.
checks = {
    "duplicate_route_ids": int(routes["route_id"].duplicated().sum()),
    "duplicate_trip_ids": int(trips["trip_id"].duplicated().sum()),
    "duplicate_stop_ids": int(stops["stop_id"].duplicated().sum()),
    "trips_missing_route": int((~trips["route_id"].isin(routes["route_id"])).sum()),
    "trips_missing_service": int((~trips["service_id"].isin(calendar["service_id"])).sum()),
    "stop_times_missing_trip": int((~stop_times["trip_id"].isin(trips["trip_id"])).sum()),
    "stop_times_missing_stop": int((~stop_times["stop_id"].isin(stops["stop_id"])).sum()),
    "duplicate_trip_stop_sequences": int(stop_times.duplicated(["trip_id", "stop_sequence"]).sum()),
}
if any(checks.values()):
    raise ValueError(f"GTFS integrity checks failed: {checks}")

lookup = (stop_times.merge(trips, on="trip_id", how="inner", validate="many_to_one")
          .merge(routes, on="route_id", how="inner", validate="many_to_one")
          .merge(stops, on="stop_id", how="inner", validate="many_to_one"))
columns = ["route_id", "route_short_name", "route_long_name", "service_id", "trip_id",
           "trip_headsign", "direction_id", "stop_sequence", "stop_id", "stop_name",
           "stop_lat", "stop_lon", "arrival_time", "departure_time"]
lookup = lookup[columns].sort_values(["route_id", "direction_id", "trip_id", "stop_sequence"], kind="stable").reset_index(drop=True)
if len(lookup) != len(stop_times) or lookup.empty:
    raise ValueError(f"Unexpected join row count: {len(lookup)} vs {len(stop_times)}")
part_b_path = PROCESSED_DIR / "route_stop_lookup.csv.gz"
part_b_path.write_bytes(gzip.compress(lookup.to_csv(index=False).encode("utf-8"), compresslevel=9, mtime=0))
output_sha256 = hashlib.sha256(part_b_path.read_bytes()).hexdigest()
print("Validation:", checks)
print("Prepared rows:", len(lookup), "output:", part_b_path, "SHA-256:", output_sha256)
display(lookup.head(5))


## Cell 4 — Write the Part B lineage manifest

In [ ]:
counts = {name: int(len(frame)) for name, frame in tables.items()}
manifest = {
    "artifact": "week3-part-b-data-preparation",
    "data_type": "structured_gtfs_tabular",
    "source_url": SOURCE_URL,
    "source_provider": "Prasarana Malaysia via Malaysia Open API",
    "license": LICENSE,
    "attribution": "Prasarana Malaysia Berhad via data.gov.my GTFS Static API",
    "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_zip_sha256": source_sha256,
    "source_zip_bytes": len(source_bytes),
    "input_row_counts": counts,
    "validation": checks,
    "processing": "Join stop_times to trips by trip_id, routes by route_id and stops by stop_id; preserve source IDs; sort deterministically.",
    "output": {"path": "data/processed/route_stop_lookup.csv.gz", "sha256": output_sha256,
               "rows": len(lookup), "columns": columns},
    "limitations": [
        "Provider reports about 2% of Rapid Bus KL trips missing from stop_times.txt due to operational issues.",
        "Calendar dates must be checked on each refresh; this snapshot is not real-time service status."
    ]
}
(BASE_DIR / "data" / "week3_manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print("Part B manifest:", BASE_DIR / "data" / "week3_manifest.json")


## Cell 5 — Define and execute the Part C V1 pipeline

In [ ]:
PIPELINE_DIR = BASE_DIR / "data" / "pipeline_v1" / "tabular"
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

def run_pipeline(source_bytes):
    source_hash = hashlib.sha256(source_bytes).hexdigest()
    with zipfile.ZipFile(BytesIO(source_bytes)) as zf:
        if zf.testzip() is not None:
            raise ValueError("ZIP integrity check failed")
        needed = ["routes.txt", "trips.txt", "stops.txt", "stop_times.txt", "calendar.txt"]
        missing = set(needed) - set(zf.namelist())
        if missing:
            raise ValueError(f"Missing required tables: {sorted(missing)}")
        t = {n: pd.read_csv(zf.open(n), dtype=str, keep_default_na=False) for n in needed}
    r, tr, st, stt, cal = (t[n] for n in needed)
    validation = {
        "duplicate_route_ids": int(r.route_id.duplicated().sum()),
        "duplicate_trip_ids": int(tr.trip_id.duplicated().sum()),
        "duplicate_stop_ids": int(st.stop_id.duplicated().sum()),
        "trips_missing_route": int((~tr.route_id.isin(r.route_id)).sum()),
        "trips_missing_service": int((~tr.service_id.isin(cal.service_id)).sum()),
        "stop_times_missing_trip": int((~stt.trip_id.isin(tr.trip_id)).sum()),
        "stop_times_missing_stop": int((~stt.stop_id.isin(st.stop_id)).sum()),
        "duplicate_trip_stop_sequences": int(stt.duplicated(["trip_id", "stop_sequence"]).sum()),
    }
    if any(validation.values()):
        raise ValueError(f"Validation failed: {validation}")
    out = (stt.merge(tr, on="trip_id", validate="many_to_one")
           .merge(r, on="route_id", validate="many_to_one")
           .merge(st, on="stop_id", validate="many_to_one"))
    out = out[columns].sort_values(["route_id", "direction_id", "trip_id", "stop_sequence"], kind="stable").reset_index(drop=True)
    if len(out) != len(stt) or out.empty:
        raise ValueError("Join output is empty or row count changed")
    output_path = PIPELINE_DIR / "prepared_route_stop_lookup.csv.gz"
    output_path.write_bytes(gzip.compress(out.to_csv(index=False).encode("utf-8"), compresslevel=9, mtime=0))
    digest = hashlib.sha256(output_path.read_bytes()).hexdigest()
    run = {"source_sha256": source_hash, "row_counts": {n: int(len(v)) for n,v in t.items()},
           "validation": validation, "output_sha256": digest, "output_rows": len(out),
           "source_output_mapping": [{"source": "rapid-bus-kl GTFS ZIP", "output": "data/pipeline_v1/tabular/prepared_route_stop_lookup.csv.gz"}]}
    return run

first_run = run_pipeline(source_bytes)
pipeline_manifest = {
    "pipeline_version": "week3-v1", "data_type": "tabular",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "source": {"url": SOURCE_URL, "license": LICENSE, "sha256": first_run["source_sha256"], "bytes": len(source_bytes)},
    "processing": "Inspect required GTFS tables; validate keys and references; join route, trip, stop-time and stop records; sort; export and fingerprint.",
    "first_run": first_run,
    "known_limitations": manifest["limitations"]
}
(PIPELINE_DIR / "pipeline_manifest.json").write_text(json.dumps(pipeline_manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print("Pipeline completed successfully.")
print("Output rows:", first_run["output_rows"])
print("Manifest:", PIPELINE_DIR / "pipeline_manifest.json")


## Cell 6 — Repeatability and provenance check

In [ ]:
second_run = run_pipeline(source_bytes)
assert first_run["source_sha256"] == second_run["source_sha256"]
assert first_run["row_counts"] == second_run["row_counts"]
assert first_run["validation"] == second_run["validation"]
assert first_run["output_sha256"] == second_run["output_sha256"]
assert first_run["source_output_mapping"] == second_run["source_output_mapping"]
print("PASS: source fingerprint, table counts, validation, prepared output hash and source-to-output mapping match.")
print("Execution timestamps are expected to differ.")


## GitHub handoff

The raw source ZIP is temporary in Colab. Download/commit only the prepared CSV, lineage manifests, notebook, inventory, pipeline diagram and decisions document. Keep the CC BY 4.0 attribution and do not claim live arrivals or complete trip coverage.